# 长期预测任务数据集探查

本 Notebook 对长期预测任务的所有数据集进行数据探查，展示归一化后的数据分布。

## 数据集划分方式

| 数据集 | 划分方式 | 训练集 | 验证集 | 测试集 |
|--------|----------|--------|--------|--------|
| ETTh1, ETTh2 | 时间划分 (12+4+4 个月) | 0-12月 | 12-16月 | 16-20月 |
| ETTm1, ETTm2 | 时间划分 (12+4+4 个月, 15分钟粒度) | 0-12月 | 12-16月 | 16-20月 |
| Custom (electricity, exchange_rate, weather...) | 比例划分 (70%+20%+10%) | 前70% | 后20% | 中间10% |

In [2]:
import sys
sys.path.insert(0, '/data/nishome/xuhaochen/Time-Series-Library/')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
import os
os.chdir('/data/nishome/xuhaochen/Time-Series-Library/')

plt.rcParams['figure.figsize'] = [12, 4]
plt.rcParams['font.size'] = 10
pd.set_option('display.max_columns', 10)

## 1. 数据集定义与划分逻辑

In [3]:
class LongTermForecastDataLoader:
    """
    长期预测任务数据加载器
    - 自动从原始数据划分训练/验证/测试集
    - 使用训练集进行归一化
    - 返回归一化后的完整数据（不分 batch）
    """
    
    # 数据集配置
    DATASET_CONFIG = {
        'ETTh1': {
            'path': './dataset/ETT-small/ETTh1.csv',
            'type': 'ETT_hour',
            'freq': 'h',
            'channels': 7
        },
        'ETTh2': {
            'path': './dataset/ETT-small/ETTh2.csv',
            'type': 'ETT_hour',
            'freq': 'h',
            'channels': 7
        },
        'ETTm1': {
            'path': './dataset/ETT-small/ETTm1.csv',
            'type': 'ETT_minute',
            'freq': 't',
            'channels': 7
        },
        'ETTm2': {
            'path': './dataset/ETT-small/ETTm2.csv',
            'type': 'ETT_minute',
            'freq': 't',
            'channels': 7
        },
        'electricity': {
            'path': './dataset/electricity/electricity.csv',
            'type': 'Custom',
            'freq': 'h',
            'channels': 321
        },
        'exchange_rate': {
            'path': './dataset/exchange_rate/exchange_rate.csv',
            'type': 'Custom',
            'freq': 'd',
            'channels': 8
        },
        'weather': {
            'path': './dataset/weather/weather.csv',
            'type': 'Custom',
            'freq': 'h',
            'channels': 21
        },
    }
    
    def __init__(self, dataset_name):
        self.dataset_name = dataset_name
        self.config = self.DATASET_CONFIG[dataset_name]
        self.scaler = StandardScaler()
        
        # 加载原始数据
        self.df_raw = pd.read_csv(self.config['path'])
        print(f"数据集: {dataset_name}")
        print(f"原始数据形状: {self.df_raw.shape}")
        print(f"时间范围: {self.df_raw['date'].iloc[0]} ~ {self.df_raw['date'].iloc[-1]}")
    
    def _get_ett_hour_borders(self, seq_len=96):
        """
        ETT_hour 数据集划分:
        - 训练: 0 ~ 12*30*24 (12个月，按小时计)
        - 验证: 12*30*24 - seq_len ~ 12*30*24 + 4*30*24
        - 测试: 12*30*24 + 4*30*24 - seq_len ~ 结束
        """
        border1s = [0, 12*30*24 - seq_len, 12*30*24 + 4*30*24 - seq_len]
        border2s = [12*30*24, 12*30*24 + 4*30*24, 12*30*24 + 8*30*24]
        return border1s, border2s
    
    def _get_ett_minute_borders(self, seq_len=96):
        """
        ETT_minute 数据集划分 (15分钟粒度):
        - 训练: 0 ~ 12*30*24*4
        - 验证: 12*30*24*4 - seq_len ~ 12*30*24*4 + 4*30*24*4
        - 测试: 12*30*24*4 + 4*30*24*4 - seq_len ~ 结束
        """
        border1s = [0, 12*30*24*4 - seq_len, 12*30*24*4 + 4*30*24*4 - seq_len]
        border2s = [12*30*24*4, 12*30*24*4 + 4*30*24*4, 12*30*24*4 + 8*30*24*4]
        return border1s, border2s
    
    def _get_custom_borders(self, seq_len=96):
        """
        Custom 数据集划分:
        - num_train = 70% (前70%)
        - num_test = 20% (后20%)
        - num_vali = 10% (剩余10%，即中间10%)
        """
        n = len(self.df_raw)
        num_train = int(n * 0.7)
        num_test = int(n * 0.2)
        num_vali = n - num_train - num_test  # = 10%
        
        border1s = [0, num_train - seq_len, n - num_test - seq_len]
        border2s = [num_train, num_train + num_vali, n]
        return border1s, border2s
    
    def load_data(self, seq_len=96):
        """
        加载并归一化数据
        
        Returns:
            dict: 包含 train/val/test 的 data_x, data_y, data_stamp
        """
        data_type = self.config['type']
        
        # 获取数据划分边界
        if data_type == 'ETT_hour':
            border1s, border2s = self._get_ett_hour_borders(seq_len)
        elif data_type == 'ETT_minute':
            border1s, border2s = self._get_ett_minute_borders(seq_len)
        else:  # Custom
            border1s, border2s = self._get_custom_borders(seq_len)
        
        print(f"\n数据划分 (seq_len={seq_len}):")
        print(f"  训练集: [{border1s[0]}, {border2s[0]}) -> {border2s[0] - border1s[0]} 条")
        print(f"  验证集: [{border1s[1]}, {border2s[1]}) -> {border2s[1] - border1s[1]} 条")
        print(f"  测试集: [{border1s[2]}, {border2s[2]}) -> {border2s[2] - border1s[2]} 条")
        
        # 提取特征列 (去掉 date)
        cols = list(self.df_raw.columns)
        cols.remove('date')
        df_data = self.df_raw[cols]
        
        # 使用训练集拟合 scaler
        train_data = df_data.iloc[border1s[0]:border2s[0]]
        self.scaler.fit(train_data.values)
        print(f"\n归一化参数 (基于训练集):")
        print(f"  均值形状: {self.scaler.mean_.shape}")
        print(f"  标准差形状: {self.scaler.scale_.shape}")
        
        # 归一化所有数据
        data_normalized = self.scaler.transform(df_data.values)
        
        # 时间戳
        df_stamp = self.df_raw[['date']]
        
        # 构建结果字典
        result = {}
        for flag, idx in zip(['train', 'val', 'test'], [0, 1, 2]):
            result[flag] = {
                'data_x': data_normalized[border1s[idx]:border2s[idx]],
                'data_y': data_normalized[border1s[idx]:border2s[idx]],
                'data_stamp': df_stamp.iloc[border1s[idx]:border2s[idx]]['date'].values,
                'border1': border1s[idx],
                'border2': border2s[idx]
            }
        
        return result

# 测试数据加载器
loader = LongTermForecastDataLoader('ETTh1')
data_dict = loader.load_data(seq_len=96)

数据集: ETTh1
原始数据形状: (17420, 8)
时间范围: 2016-07-01 00:00:00 ~ 2018-06-26 19:00:00

数据划分 (seq_len=96):
  训练集: [0, 8640) -> 8640 条
  验证集: [8544, 11520) -> 2976 条
  测试集: [11424, 14400) -> 2976 条

归一化参数 (基于训练集):
  均值形状: (7,)
  标准差形状: (7,)


## 2. 探查所有数据集

In [6]:
# 定义所有长期预测数据集
LONG_TERM_DATASETS = ['ETTh1', 'ETTh2', 'ETTm1', 'ETTm2', 'electricity', 'exchange_rate', 'weather']

# 加载所有数据集
all_data = {}
for name in LONG_TERM_DATASETS:
    print(f"\n{'='*60}")
    loader = LongTermForecastDataLoader(name)
    all_data[name] = loader.load_data(seq_len=96)
    #print(f"  归一化后数据形状: {all_data[name]['train']['data_x'].shape}")


数据集: ETTh1
原始数据形状: (17420, 8)
时间范围: 2016-07-01 00:00:00 ~ 2018-06-26 19:00:00

数据划分 (seq_len=96):
  训练集: [0, 8640) -> 8640 条
  验证集: [8544, 11520) -> 2976 条
  测试集: [11424, 14400) -> 2976 条

归一化参数 (基于训练集):
  均值形状: (7,)
  标准差形状: (7,)

数据集: ETTh2
原始数据形状: (17420, 8)
时间范围: 2016-07-01 00:00:00 ~ 2018-06-26 19:00:00

数据划分 (seq_len=96):
  训练集: [0, 8640) -> 8640 条
  验证集: [8544, 11520) -> 2976 条
  测试集: [11424, 14400) -> 2976 条

归一化参数 (基于训练集):
  均值形状: (7,)
  标准差形状: (7,)

数据集: ETTm1
原始数据形状: (69680, 8)
时间范围: 2016-07-01 00:00:00 ~ 2018-06-26 19:45:00

数据划分 (seq_len=96):
  训练集: [0, 34560) -> 34560 条
  验证集: [34464, 46080) -> 11616 条
  测试集: [45984, 57600) -> 11616 条

归一化参数 (基于训练集):
  均值形状: (7,)
  标准差形状: (7,)

数据集: ETTm2
原始数据形状: (69680, 8)
时间范围: 2016-07-01 00:00:00 ~ 2018-06-26 19:45:00

数据划分 (seq_len=96):
  训练集: [0, 34560) -> 34560 条
  验证集: [34464, 46080) -> 11616 条
  测试集: [45984, 57600) -> 11616 条

归一化参数 (基于训练集):
  均值形状: (7,)
  标准差形状: (7,)

数据集: electricity
原始数据形状: (26304, 322)
时间范围: 2016-07-01 02:00:

## 3. 各数据集统计信息

In [5]:
# 汇总统计表
stats = []
for name in LONG_TERM_DATASETS:
    config = LongTermForecastDataLoader.DATASET_CONFIG[name]
    train_data = all_data[name]['train']['data_x']
    val_data = all_data[name]['val']['data_x']
    test_data = all_data[name]['test']['data_x']
    
    stats.append({
        '数据集': name,
        '类型': config['type'],
        '通道数': config['channels'],
        '频率': config['freq'],
        '训练集大小': train_data.shape[0],
        '验证集大小': val_data.shape[0],
        '测试集大小': test_data.shape[0],
    })

stats_df = pd.DataFrame(stats)
stats_df

,数据集,类型,通道数,频率,训练集大小,验证集大小,测试集大小
0,ETTh1,ETT_hour,7,h,8640,2976,2976
1,ETTh2,ETT_hour,7,h,8640,2976,2976
2,ETTm1,ETT_minute,7,t,34560,11616,11616
3,ETTm2,ETT_minute,7,t,34560,11616,11616
4,electricity,Custom,321,h,18412,2728,5356
5,exchange_rate,Custom,8,d,5311,856,1613
6,weather,Custom,21,h,36887,5366,10635


In [ ]:
# 打印详细统计
for name in LONG_TERM_DATASETS:
    train_data = all_data[name]['train']['data_x']
    print(f"\n{name}:")
    print(f"  训练集形状: {train_data.shape}")
    print(f"  训练集均值范围: [{train_data.mean(axis=0).min():.4f}, {train_data.mean(axis=0).max():.4f}]")
    print(f"  训练集标准差范围: [{train_data.std(axis=0).min():.4f}, {train_data.std(axis=0).max():.4f}]")
    print(f"  训练集最小值: {train_data.min():.4f}")
    print(f"  训练集最大值: {train_data.max():.4f}")

## 4. 数据分布可视化

In [ ]:
# 绘制各数据集训练集的分布 (第一个通道)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for idx, name in enumerate(LONG_TERM_DATASETS):
    train_data = all_data[name]['train']['data_x']
    axes[idx].hist(train_data[:, 0], bins=50, edgecolor='black', alpha=0.7)
    axes[idx].set_title(f'{name}\n(通道0, n={train_data.shape[0]})')
    axes[idx].set_xlabel('值 (归一化后)')
    axes[idx].set_ylabel('频数')

# 隐藏多余的子图
for idx in range(len(LONG_TERM_DATASETS), len(axes)):
    axes[idx].axis('off')

plt.tight_layout()
plt.savefig('./0notebook/dataset_distribution.png', dpi=150)
plt.show()

In [ ]:
# 绘制时间序列示例 (每个数据集的第一个通道)
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
axes = axes.flatten()

for idx, name in enumerate(LONG_TERM_DATASETS):
    train_data = all_data[name]['train']['data_x']
    # 只显示前1000个点
    n_show = min(1000, train_data.shape[0])
    axes[idx].plot(train_data[:n_show, 0])
    axes[idx].set_title(f'{name}\n(通道0, 前{n_show}点)')
    axes[idx].set_xlabel('时间步')
    axes[idx].set_ylabel('值')

for idx in range(len(LONG_TERM_DATASETS), len(axes)):
    axes[idx].axis('off')

plt.tight_layout()
plt.savefig('./0notebook/dataset_timeseries.png', dpi=150)
plt.show()

## 5. 各通道统计热力图

In [ ]:
# 绘制通道统计热力图 (ETTh1)
name = 'ETTh1'
train_data = all_data[name]['train']['data_x']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 均值
im0 = axes[0].imshow(train_data.mean(axis=0).reshape(1, -1), aspect='auto', cmap='coolwarm')
axes[0].set_title(f'{name} - 各通道均值')
axes[0].set_ylabel('Channels')
axes[0].set_xlabel('通道索引')
plt.colorbar(im0, ax=axes[0])

# 标准差
im1 = axes[1].imshow(train_data.std(axis=0).reshape(1, -1), aspect='auto', cmap='coolwarm')
axes[1].set_title(f'{name} - 各通道标准差')
axes[1].set_ylabel('Channels')
axes[1].set_xlabel('通道索引')
plt.colorbar(im1, ax=axes[1])

# 最大值
im2 = axes[2].imshow(train_data.max(axis=0).reshape(1, -1), aspect='auto', cmap='coolwarm')
axes[2].set_title(f'{name} - 各通道最大值')
axes[2].set_ylabel('Channels')
axes[2].set_xlabel('通道索引')
plt.colorbar(im2, ax=axes[2])

plt.tight_layout()
plt.savefig('./0notebook/ETTh1_channel_stats.png', dpi=150)
plt.show()

In [ ]:
# 绘制 weather 的通道统计
name = 'weather'
train_data = all_data[name]['train']['data_x']

fig, axes = plt.subplots(1, 3, figsize=(15, 3))

im0 = axes[0].imshow(train_data.mean(axis=0).reshape(1, -1), aspect='auto', cmap='coolwarm')
axes[0].set_title(f'{name} - 各通道均值')
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(train_data.std(axis=0).reshape(1, -1), aspect='auto', cmap='coolwarm')
axes[1].set_title(f'{name} - 各通道标准差')
plt.colorbar(im1, ax=axes[1])

im2 = axes[2].imshow(train_data.max(axis=0).reshape(1, -1), aspect='auto', cmap='coolwarm')
axes[2].set_title(f'{name} - 各通道最大值')
plt.colorbar(im2, ax=axes[2])

plt.tight_layout()
plt.savefig('./0notebook/weather_channel_stats.png', dpi=150)
plt.show()

## 6. 数据集划分方式详解

In [ ]:
# 可视化数据划分方式
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# ETT_hour 划分
ax = axes[0]
border1s, border2s = LongTermForecastDataLoader('ETTh1')._get_ett_hour_borders(seq_len=96)
total_len = border2s[2]
ax.barh(0, border2s[0]-border1s[0], left=border1s[0], color='blue', alpha=0.7, label='训练集')
ax.barh(0, border2s[1]-border1s[1], left=border1s[1], color='orange', alpha=0.7, label='验证集')
ax.barh(0, border2s[2]-border1s[2], left=border1s[2], color='green', alpha=0.7, label='测试集')
ax.set_xlim(0, total_len)
ax.set_yticks([])
ax.set_xlabel('时间步 (小时)')
ax.set_title('ETTh1/ETTh2 划分方式\n(时间划分: 12+4+4 个月)')
ax.legend(loc='upper right')

# ETT_minute 划分
ax = axes[1]
border1s, border2s = LongTermForecastDataLoader('ETTm1')._get_ett_minute_borders(seq_len=96)
total_len = border2s[2]
ax.barh(0, border2s[0]-border1s[0], left=border1s[0], color='blue', alpha=0.7, label='训练集')
ax.barh(0, border2s[1]-border1s[1], left=border1s[1], color='orange', alpha=0.7, label='验证集')
ax.barh(0, border2s[2]-border1s[2], left=border1s[2], color='green', alpha=0.7, label='测试集')
ax.set_xlim(0, total_len)
ax.set_yticks([])
ax.set_xlabel('时间步 (15分钟)')
ax.set_title('ETTm1/ETTm2 划分方式\n(时间划分: 12+4+4 个月, 15分钟粒度)')

# Custom 划分 (electricity)
ax = axes[2]
df_temp = pd.read_csv('./dataset/electricity/electricity.csv')
n = len(df_temp)
num_train = int(n * 0.7)
num_test = int(n * 0.2)
num_vali = n - num_train - num_test
border1s = [0, num_train - 96, n - num_test - 96]
border2s = [num_train, num_train + num_vali, n]
total_len = n
ax.barh(0, border2s[0]-border1s[0], left=border1s[0], color='blue', alpha=0.7, label='训练集')
ax.barh(0, border2s[1]-border1s[1], left=border1s[1], color='orange', alpha=0.7, label='验证集')
ax.barh(0, border2s[2]-border1s[2], left=border1s[2], color='green', alpha=0.7, label='测试集')
ax.set_xlim(0, total_len)
ax.set_yticks([])
ax.set_xlabel('时间步')
ax.set_title('Custom (electricity/weather/exchange_rate)\n(比例划分: 70%+20%+10%)')

plt.tight_layout()
plt.savefig('./0notebook/data_split_comparison.png', dpi=150)
plt.show()

## 7. 获取模型输入格式数据

In [ ]:
# 演示如何获取模型接收到的数据格式
# 模型输入: seq_x (B, seq_len, channels), seq_y (B, label_len+pred_len, channels)

def get_model_input(data_dict, seq_len=96, label_len=48, pred_len=48):
    """
    将完整数据转换为模型输入格式
    
    Args:
        data_dict: LongTermForecastDataLoader.load_data() 返回的数据
        seq_len: 输入序列长度
        label_len: 标签长度 (解码器看到的真实值长度)
        pred_len: 预测长度
    
    Returns:
        X: (num_samples, seq_len, channels) 输入序列
        Y: (num_samples, label_len+pred_len, channels) 目标序列
    """
    data_x = data_dict['data_x']
    n = len(data_x) - seq_len - pred_len + 1
    
    X = np.zeros((n, seq_len, data_x.shape[1]))
    Y = np.zeros((n, label_len + pred_len, data_x.shape[1]))
    
    for i in range(n):
        s_begin = i
        s_end = s_begin + seq_len
        r_begin = s_end - label_len
        r_end = r_begin + label_len + pred_len
        
        X[i] = data_x[s_begin:s_end]
        Y[i] = data_x[r_begin:r_end]
    
    return X, Y

# 测试 - 以 ETTh1 为例
name = 'ETTh1'
X_train, Y_train = get_model_input(all_data[name]['train'], seq_len=96, label_len=48, pred_len=48)
X_val, Y_val = get_model_input(all_data[name]['val'], seq_len=96, label_len=48, pred_len=48)
X_test, Y_test = get_model_input(all_data[name]['test'], seq_len=96, label_len=48, pred_len=48)

print(f"数据集: {name}")
print(f"训练集:")
print(f"  X_train 形状: {X_train.shape}  (样本数, seq_len, 通道数)")
print(f"  Y_train 形状: {Y_train.shape}  (样本数, label_len+pred_len, 通道数)")
print(f"验证集:")
print(f"  X_val 形状: {X_val.shape}")
print(f"  Y_val 形状: {Y_val.shape}")
print(f"测试集:")
print(f"  X_test 形状: {X_test.shape}")
print(f"  Y_test 形状: {Y_test.shape}")

In [ ]:
# 验证模型输入数据
print("模型输入数据验证 (第一个样本的第一个通道):")
print(f"X_train[0, :, 0]:\n{X_train[0, :, 0]}")
print(f"\nY_train[0, :, 0]:\n{Y_train[0, :, 0]}")
print(f"\n说明: Y 的前48个值对应 label_len, 后48个值对应 pred_len (预测目标)")

## 8. 总结

### 数据划分方式

1. **ETT 数据集 (ETTh1, ETTh2, ETTm1, ETTm2)**
   - 按时间顺序划分：12个月训练 + 4个月验证 + 4个月测试
   - 每个时间步对应固定的物理时间 (小时或15分钟)

2. **Custom 数据集 (electricity, exchange_rate, weather)**
   - 按比例划分：70% 训练 + 20% 测试 + 10% 验证
   - 数据已按时间排序

### 归一化
   - 使用训练集的均值和标准差进行 StandardScaler 归一化
   - 归一化后的数据均值接近 0，标准差接近 1

### 模型输入格式
   - X: (num_samples, seq_len, channels) - 输入序列
   - Y: (num_samples, label_len+pred_len, channels) - 目标序列 (含label_len个真实值作为解码器输入)